In [1]:
from commonroad.common.file_reader import CommonRoadFileReader
from commonroad.common.file_writer import CommonRoadFileWriter
from commonroad.common.util import FileFormat
from commonroad.common.writer.file_writer_interface import OverwriteExistingFile

#  Reading an xml file and writing it in a protobuf format

#  File path of an xml file
file_path_arg = "USA_Lanker-1_1_T-1.xml"

# Reading the scenario and the problem set with the CommonRoad xml file reader
scenario_arg, planning_problem_set_arg = CommonRoadFileReader(file_path_arg).open()

# Creating a CommonRoad protobuf file writer
fw_arg = CommonRoadFileWriter(
    scenario_arg, planning_problem_set_arg, file_format=FileFormat.PROTOBUF
)

# Declaring the path and the names of the new protobuf files
filename_arg_map = "./protobuf_files/USA_Lanker-1.pb"
filename_arg_dynamic = "./protobuf_files/USA_Lanker-1_1_T-1.pb"
filename_arg_scenario = "./protobuf_files/USA_Lanker-1_1_T-1-SC.pb"

# Writing protobuf files. In this example, they are being written in the 'protobuf_files' folder.
fw_arg.write_map_to_file(filename_arg_map, overwrite_existing_file=OverwriteExistingFile.ALWAYS)
fw_arg.write_dynamic_to_file(
    filename_arg_dynamic, overwrite_existing_file=OverwriteExistingFile.ALWAYS
)
fw_arg.write_scenario_to_file(
    filename_arg_scenario, overwrite_existing_file=OverwriteExistingFile.ALWAYS
)

Replace file ./protobuf_files/USA_Lanker-1.pb
Replace file ./protobuf_files/USA_Lanker-1_1_T-1.pb
Replace file ./protobuf_files/USA_Lanker-1_1_T-1-SC.pb


In [2]:
from commonroad.common.file_reader import CommonRoadFileReader

#  Reading protobuf files

#  File paths of map, dynamic and scenario protobuf files
pb_map_file_path = "./protobuf_files/USA_Lanker-1.pb"
pb_dynamic_file_path = "./protobuf_files/USA_Lanker-1_1_T-1.pb"
pb_scenario_file_path = "./protobuf_files/USA_Lanker-1_1_T-1-SC.pb"

# creating a reader
reader = CommonRoadFileReader(
    filename_map=pb_map_file_path,
    filename_dynamic=pb_dynamic_file_path,
    filename_scenario=pb_scenario_file_path,
)

#  Reading a protobuf map file. Map_pb is read as a CommonRoad lanelet network.
map_pb = reader.open_map()

#  Reading a protobuf scenario file
scenario_pb = reader.open_scenario()

#  Reading a protobuf dynamic file
dynamic_pb = reader.open_dynamic()

In [3]:
"""
Upon reading the individual protobuf files, we can combine them to get the desired CommonRoad classes.
Pay attention to the order of the protobuf files inside the functions.
"""

#  Combining map and dynamic protobuf files into a CommonRoad scenario file
scenario_cr = reader.open_map_dynamic()

#  Combining map, dynamic and scenario protobuf files into a CommonRoad scenario file and a set of planning problems
scenario, planning_problem_set, _ = reader.open_all()

In [4]:
from commonroad.common.file_writer import CommonRoadFileWriter, OverwriteExistingFile
from commonroad.common.util import FileFormat
from commonroad.scenario.scenario import Tag

type(scenario), type(planning_problem_set)

tags = {Tag.URBAN} if not scenario.tags else scenario.tags
fw = CommonRoadFileWriter(
    scenario,
    planning_problem_set,
    "author",
    "affiliation",
    "source",
    tags=tags,
    file_format=FileFormat.XML,
)
fw.write_to_file("./protobuf_writing_test.xml", OverwriteExistingFile.ALWAYS)

Replace file ./protobuf_writing_test.xml


/home/andrew/Documents/TUM_Workspace/commonroad-io/commonroad/common/writer/file_writer_xml.py:497: UserWarning: <CommonRoadFileWriter/lanelet.lanelet_type> Lanelet 3419 has no lanelet type! Default lanelet type is used!
  warnings.warn(
/home/andrew/Documents/TUM_Workspace/commonroad-io/commonroad/common/writer/file_writer_xml.py:497: UserWarning: <CommonRoadFileWriter/lanelet.lanelet_type> Lanelet 3432 has no lanelet type! Default lanelet type is used!
  warnings.warn(
/home/andrew/Documents/TUM_Workspace/commonroad-io/commonroad/common/writer/file_writer_xml.py:497: UserWarning: <CommonRoadFileWriter/lanelet.lanelet_type> Lanelet 3440 has no lanelet type! Default lanelet type is used!
  warnings.warn(
/home/andrew/Documents/TUM_Workspace/commonroad-io/commonroad/common/writer/file_writer_xml.py:497: UserWarning: <CommonRoadFileWriter/lanelet.lanelet_type> Lanelet 3666 has no lanelet type! Default lanelet type is used!
  warnings.warn(
/home/andrew/Documents/TUM_Workspace/commonroad-

In [5]:
import numpy as np

from commonroad.common.file_reader import CommonRoadFileReader

EPS = 1e-6


def allclose(a, b):
    a, b = np.asarray(a), np.asarray(b)
    return a.shape == b.shape and np.allclose(a, b, atol=EPS)


def values_equal(a, b):
    if isinstance(a, np.ndarray) or isinstance(b, np.ndarray):
        return allclose(a, b)
    if isinstance(a, float):
        return abs(a - b) <= EPS
    return a == b


def same_dynamic_obstacle(a, b):
    if a.obstacle_type != b.obstacle_type:
        return False
    if a.obstacle_shape != b.obstacle_shape:
        return False

    ta = a.prediction.trajectory.state_list
    tb = b.prediction.trajectory.state_list

    if len(ta) != len(tb):
        return False

    for sa, sb in zip(ta, tb):
        if not np.allclose(sa.position, sb.position, atol=EPS):
            return False

    return True


def diff_commonroad(label_a, scn_a, pps_a, label_b, scn_b, pps_b):
    diffs = []

    def add(msg):
        diffs.append(f"[{label_a} vs {label_b}] {msg}")

    # ── Lanelets ─────────────────────────────────────────
    net_a, net_b = scn_a.lanelet_network, scn_b.lanelet_network
    ids_a = {lane.lanelet_id for lane in net_a.lanelets}
    ids_b = {lane.lanelet_id for lane in net_b.lanelets}

    for i in ids_a ^ ids_b:
        add(f"Lanelet {'removed' if i in ids_a else 'added'}: {i}")

    for i in ids_a & ids_b:
        a, b = net_a.find_lanelet_by_id(i), net_b.find_lanelet_by_id(i)

        for name in ("left_vertices", "right_vertices", "center_vertices"):
            if not allclose(getattr(a, name), getattr(b, name)):
                add(f"Lanelet {i} {name} geometry differs")

        for name in ("predecessor", "successor", "adj_left", "adj_right"):
            if getattr(a, name) != getattr(b, name):
                add(f"Lanelet {i} {name}: {getattr(a, name)} → {getattr(b, name)}")

    # ── Obstacles ────────────────────────────────────────
    def diff_obstacles(tag, A, B):
        A = {o.obstacle_id: o for o in A}
        B = {o.obstacle_id: o for o in B}

        for i in A.keys() ^ B.keys():
            add(f"{tag} {'removed' if i in A else 'added'}: {i}")

        for i in A.keys() & B.keys():
            if tag == "DynamicObstacle":
                if not same_dynamic_obstacle(A[i], B[i]):
                    add(f"{tag} {i} differs")
            else:
                if A[i] != B[i]:
                    add(f"{tag} {i} differs")

    diff_obstacles("DynamicObstacle", scn_a.dynamic_obstacles, scn_b.dynamic_obstacles)
    diff_obstacles("StaticObstacle", scn_a.static_obstacles, scn_b.static_obstacles)

    # ── Planning Problems ────────────────────────────────
    A, B = pps_a.planning_problem_dict, pps_b.planning_problem_dict

    for i in A.keys() ^ B.keys():
        add(f"PlanningProblem {'removed' if i in A else 'added'}: {i}")

    for i in A.keys() & B.keys():
        if A[i].initial_state != B[i].initial_state:
            add(f"PlanningProblem {i} initial_state differs")
        if A[i].goal != B[i].goal:
            add(f"PlanningProblem {i} goal differs")

    return diffs


# ── Read all three representations ──────────────────────

scn_xml1, pps_xml1 = CommonRoadFileReader("USA_Lanker-1_1_T-1.xml").open()

reader_pb = CommonRoadFileReader(
    filename_map="./protobuf_files/USA_Lanker-1.pb",
    filename_dynamic="./protobuf_files/USA_Lanker-1_1_T-1.pb",
    filename_scenario="./protobuf_files/USA_Lanker-1_1_T-1-SC.pb",
)
scn_pb, pps_pb, _ = reader_pb.open_all()

scn_xml2, pps_xml2 = CommonRoadFileReader("./protobuf_writing_test.xml").open()


# ── Pairwise comparison ─────────────────────────────────
diffs = []
diffs += diff_commonroad("XML(orig)", scn_xml1, pps_xml1, "PB", scn_pb, pps_pb)
diffs += diff_commonroad("PB", scn_pb, pps_pb, "XML(rewrite)", scn_xml2, pps_xml2)
diffs += diff_commonroad("XML(orig)", scn_xml1, pps_xml1, "XML(rewrite)", scn_xml2, pps_xml2)

print("\n".join(diffs) if diffs else "All three representations are semantically identical.")

All three representations are semantically identical.
